# Station Data Cleaning – Objective & Plan

## Objective
Build a canonical station mapping so station-level metrics are not fragmented by year-based naming inconsistencies.

## Plan (one task at a time)
1. **Task 1 (now):** Identify identical GPS coordinates with multiple station names and quantify impact.
2. Task 2: Normalize names (trim, collapse spaces, punctuation/case normalization) and compare variant reduction.
3. Task 3: Create canonical station IDs by coordinate pair and generate `station_canonical_mapping` table.
4. Task 4: Validate mapping quality (self-loop reduction, station count consolidation, unresolved conflicts).

> Per your instruction, I will only run Task 1 now and wait for your explicit approval before Task 2.

In [1]:
import duckdb
import pandas as pd

PARQUET_GLOB = "data/**/*.parquet"

query = """
WITH stations AS (
    SELECT
        ROUND(CAST(start_station_latitude AS DOUBLE), 6) AS lat,
        ROUND(CAST(start_station_longitude AS DOUBLE), 6) AS lon,
        start_station_name AS station_name,
        EXTRACT(YEAR FROM to_timestamp(start_time_ms / 1000.0)) AS trip_year
    FROM read_parquet(?)
    WHERE start_station_latitude IS NOT NULL
      AND start_station_longitude IS NOT NULL
      AND start_station_name IS NOT NULL
)
SELECT
    lat,
    lon,
    COUNT(DISTINCT station_name) AS num_name_variants,
    COUNT(*) AS total_trips,
    LIST(DISTINCT station_name ORDER BY station_name) AS name_variants,
    MIN(trip_year) AS first_year_seen,
    MAX(trip_year) AS last_year_seen
FROM stations
GROUP BY lat, lon
HAVING COUNT(DISTINCT station_name) > 1
ORDER BY total_trips DESC, num_name_variants DESC
"""

variant_locations_df = duckdb.sql(query, params=[PARQUET_GLOB]).to_df()

print(f"Locations with multiple names at identical coordinates: {len(variant_locations_df):,}")
print(f"Trips affected (sum across these locations): {variant_locations_df['total_trips'].sum():,}")
variant_locations_df.head(20)

Locations with multiple names at identical coordinates: 179
Trips affected (sum across these locations): 3,281,706


,lat,lon,num_name_variants,total_trips,name_variants,first_year_seen,last_year_seen
0,45.520626,-73.575951,2,97794,"[Duluth / St-Denis, Duluth / St-Denis]",2024,2026
1,45.510750,-73.565125,2,94039,[Métro St-Laurent ( de Maisonneuve / St-Lauren...,2024,2026
2,45.525513,-73.574242,2,80729,"[du Parc-La Fontaine / Rachel, du Parc-Lafonta...",2024,2026
3,45.534134,-73.573524,2,78170,"[Chabot / Mont-Royal, Chabot / du Mont-Royal]",2024,2025
4,45.515869,-73.560081,3,74198,[Parc Émilie-Gamelin (St-Hubert / de Maisonneu...,2024,2026
5,45.529709,-73.569984,2,68456,"[s Émile-Duployé / Rachel, Émile-Duployé / Rac...",2024,2026
6,45.530087,-73.576881,2,68295,"[de Lanaudière / Mont-Royal, de Lanaudière / d...",2024,2025
7,45.545189,-73.576439,3,59797,"[Parc du Pélican (1ere Ave / Masson), Parc du ...",2024,2025
8,45.530754,-73.576180,2,58316,"[Garnier / Mont-Royal, Garnier / du Mont-Royal]",2024,2026
9,45.502293,-73.573303,2,56970,"[REM McGill (de Maisonneuve / Mansfield), de M...",2024,2026


In [2]:
# Task 2: Normalize station names and compare variant reduction

norm_query = """
WITH stations AS (
    SELECT
        ROUND(CAST(start_station_latitude AS DOUBLE), 6) AS lat,
        ROUND(CAST(start_station_longitude AS DOUBLE), 6) AS lon,
        start_station_name AS raw_name,
        LOWER(
            TRIM(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(
                        REGEXP_REPLACE(start_station_name, '\\s+', ' ', 'g'),
                        '\\s*/\\s*', ' / ', 'g'
                    ),
                    '\\s*-\\s*', '-', 'g'
                )
            )
        ) AS normalized_name
    FROM read_parquet(?)
    WHERE start_station_latitude IS NOT NULL
      AND start_station_longitude IS NOT NULL
      AND start_station_name IS NOT NULL
), by_coord AS (
    SELECT
        lat,
        lon,
        COUNT(DISTINCT raw_name) AS raw_variants,
        COUNT(DISTINCT normalized_name) AS normalized_variants,
        COUNT(*) AS total_trips,
        LIST(DISTINCT raw_name ORDER BY raw_name) AS raw_names,
        LIST(DISTINCT normalized_name ORDER BY normalized_name) AS normalized_names
    FROM stations
    GROUP BY lat, lon
)
SELECT
    lat,
    lon,
    raw_variants,
    normalized_variants,
    (raw_variants - normalized_variants) AS variants_reduced,
    total_trips,
    raw_names,
    normalized_names
FROM by_coord
WHERE raw_variants > 1
ORDER BY variants_reduced DESC, total_trips DESC
"""

normalization_impact_df = duckdb.sql(norm_query, params=[PARQUET_GLOB]).to_df()

raw_problem_locations = int((normalization_impact_df['raw_variants'] > 1).sum())
post_norm_problem_locations = int((normalization_impact_df['normalized_variants'] > 1).sum())
locations_fully_resolved = int((normalization_impact_df['normalized_variants'] == 1).sum())

raw_total_variants = int(normalization_impact_df['raw_variants'].sum())
norm_total_variants = int(normalization_impact_df['normalized_variants'].sum())

print(f"Raw multi-name coordinate locations: {raw_problem_locations:,}")
print(f"Post-normalization multi-name locations: {post_norm_problem_locations:,}")
print(f"Locations fully resolved by normalization: {locations_fully_resolved:,}")
print(f"Total variants (raw -> normalized): {raw_total_variants:,} -> {norm_total_variants:,}")

normalization_impact_df.head(20)

Raw multi-name coordinate locations: 179
Post-normalization multi-name locations: 141
Locations fully resolved by normalization: 38
Total variants (raw -> normalized): 371 -> 329


,lat,lon,raw_variants,normalized_variants,variants_reduced,total_trips,raw_names,normalized_names
0,45.569996,-73.573074,3,1,2,10625,"[35e avenue / Beaubien, 35e avenue/ Beaubien,...",[35e avenue / beaubien]
1,45.520626,-73.575951,2,1,1,97794,"[Duluth / St-Denis, Duluth / St-Denis]",[duluth / st-denis]
2,45.521618,-73.570374,2,1,1,52526,"[Roy / St-André, Roy/St-André]",[roy / st-andré]
3,45.507378,-73.614861,3,2,1,48966,"[Edouard-Montpetit / de Stirling, Edouard-Mont...","[edouard-montpetit / de stirling, édouard-mont..."
4,45.502087,-73.562943,2,1,1,47679,"[Square-Victoria (Viger / du Square-Victoria),...",[square-victoria (viger / du square-victoria)]
5,45.488300,-73.568718,2,1,1,36950,"[Notre-Dame / St-Martin, Notre-Dame / St-Martin]",[notre-dame / st-martin]
6,45.514011,-73.565987,2,1,1,36700,[CÉGEP du Vieux-Montréal (Ontario / de l'Hôtel...,[cégep du vieux-montréal (ontario / de l'hôtel...
7,45.521484,-73.596848,2,1,1,32908,"[de L'Esplanade / Fairmount, de l'Esplanade ...",[de l'esplanade / fairmount]
8,45.516930,-73.589119,2,1,1,31882,"[Parc Jeanne-Mance (du Mont-Royal / du Parc), ...",[parc jeanne-mance (du mont-royal / du parc)]
9,45.492107,-73.562462,2,1,1,31445,"[du Séminaire / de la Montagne, du Séminaire ...",[du séminaire / de la montagne]


In [19]:
# Task 2b: Fuzzy matching for unresolved normalized names (ratio + partial ratio + token overlap)
from difflib import SequenceMatcher
from itertools import combinations
import re

RATIO_THRESHOLD = 0.80
PARTIAL_RATIO_THRESHOLD = 0.80
TOKEN_OVERLAP_THRESHOLD = 0.85


def full_ratio(a: str, b: str) -> float:
    return SequenceMatcher(None, a, b).ratio()


def partial_ratio(a: str, b: str) -> float:
    a = a.strip().lower()
    b = b.strip().lower()

    if not a or not b:
        return 0.0

    shorter, longer = (a, b) if len(a) <= len(b) else (b, a)

    if shorter in longer:
        return 1.0

    window = len(shorter)
    best = 0.0
    for start_idx in range(0, len(longer) - window + 1):
        chunk = longer[start_idx:start_idx + window]
        score = SequenceMatcher(None, shorter, chunk).ratio()
        if score > best:
            best = score
    return best


def token_overlap_coeff(a: str, b: str) -> float:
    token_pattern = r"[a-z0-9]+"
    tokens_a = set(re.findall(token_pattern, a.lower()))
    tokens_b = set(re.findall(token_pattern, b.lower()))

    if not tokens_a or not tokens_b:
        return 0.0

    overlap = len(tokens_a & tokens_b)
    return overlap / min(len(tokens_a), len(tokens_b))


unresolved_df = normalization_impact_df[normalization_impact_df['normalized_variants'] > 1].copy()

pair_rows = []
skipped_missing = 0
skipped_too_short = 0

for _, row in unresolved_df.iterrows():
    names_raw = row['normalized_names']

    if names_raw is None:
        skipped_missing += 1
        continue

    if isinstance(names_raw, str):
        names = [names_raw]
    elif hasattr(names_raw, 'tolist'):
        names = names_raw.tolist()
    else:
        names = list(names_raw)

    names = [str(name).strip().lower() for name in names if str(name).strip()]
    names = sorted(set(names))

    if len(names) < 2:
        skipped_too_short += 1
        continue

    for left_name, right_name in combinations(names, 2):
        ratio_score = full_ratio(left_name, right_name)
        partial_score = partial_ratio(left_name, right_name)
        overlap_score = token_overlap_coeff(left_name, right_name)

        if (
            ratio_score >= RATIO_THRESHOLD
            or partial_score >= PARTIAL_RATIO_THRESHOLD
            or overlap_score >= TOKEN_OVERLAP_THRESHOLD
        ):
            pair_rows.append(
                {
                    'lat': row['lat'],
                    'lon': row['lon'],
                    'left_name': left_name,
                    'right_name': right_name,
                    'ratio_score': round(ratio_score, 4),
                    'partial_ratio_score': round(partial_score, 4),
                    'token_overlap_score': round(overlap_score, 4),
                    'max_score': round(max(ratio_score, partial_score, overlap_score), 4),
                    'total_trips_at_location': row['total_trips'],
                }
            )

fuzzy_candidates_df = pd.DataFrame(pair_rows)

print(f"Unresolved locations checked: {len(unresolved_df):,}")
print(f"Skipped (missing names): {skipped_missing:,}")
print(f"Skipped (<2 usable names): {skipped_too_short:,}")
print(f"Thresholds -> ratio: {RATIO_THRESHOLD}, partial_ratio: {PARTIAL_RATIO_THRESHOLD}, token_overlap: {TOKEN_OVERLAP_THRESHOLD}")

if fuzzy_candidates_df.empty:
    print("No candidates found with combined matching.")
else:
    locations_with_candidates = fuzzy_candidates_df[['lat', 'lon']].drop_duplicates().shape[0]
    print(f"Locations with candidate pairs: {locations_with_candidates:,}")
    print(f"Candidate name pairs found: {len(fuzzy_candidates_df):,}")
    display(
        fuzzy_candidates_df
        .sort_values(['max_score', 'total_trips_at_location'], ascending=[False, False])
        .head(50)
    )

Unresolved locations checked: 141
Skipped (missing names): 0
Skipped (<2 usable names): 0
Thresholds -> ratio: 0.8, partial_ratio: 0.8, token_overlap: 0.85
Locations with candidate pairs: 122
Candidate name pairs found: 134


,lat,lon,left_name,right_name,ratio_score,partial_ratio_score,token_overlap_score,max_score,total_trips_at_location
3,45.510750,-73.565125,métro st-laurent ( de maisonneuve / st-laurent),métro st-laurent (de maisonneuve / st-laurent),0.9892,0.9783,1.0,1.0,94039
5,45.534134,-73.573524,chabot / du mont-royal,chabot / mont-royal,0.9268,0.8421,1.0,1.0,78170
7,45.515869,-73.560081,parc émilie-gamelin (st-hubert / de maisonneuv...,st-hubert / de maisonneuve (sud),0.7381,0.9688,1.0,1.0,74198
9,45.529709,-73.569984,s émile-duployé / rachel,émile-duployé / rachel,0.9565,1.0000,1.0,1.0,68456
10,45.530087,-73.576881,de lanaudière / du mont-royal,de lanaudière / mont-royal,0.9455,0.8846,1.0,1.0,68295
14,45.530754,-73.576180,garnier / du mont-royal,garnier / mont-royal,0.9302,0.8500,1.0,1.0,58316
15,45.520500,-73.567703,st-andré / cherrier,st-andré / cherrier_swap eco 5,0.7755,1.0000,1.0,1.0,50826
16,45.531921,-73.598045,de st-vallier / rosemont,métro rosemont (de st-vallier / rosemont),0.7385,1.0000,1.0,1.0,48666
17,45.537086,-73.593216,chambord / rosemont,parc père-marquette (chambord / rosemont),0.6333,1.0000,1.0,1.0,43850
20,45.539307,-73.595222,de bellechasse / de lanaudière,parc du père-marquette (de bellechasse / de la...,0.7059,1.0000,1.0,1.0,39700


## Pre-Task 3 Sub-analysis: Similar Station Names at Different Coordinates

### Objective
Identify station names that are textually similar but map to **different** `(lat, lon)` locations.

### Plan
1. Build one primary coordinate per normalized station name (most-used coordinate).
2. Compare station names pairwise using `ratio`, `partial_ratio`, and token overlap.
3. Keep high-similarity pairs where coordinates differ and compute distance (km).

In [3]:
# Similar names at different coordinates (before Task 3) - optimized
from itertools import combinations
from math import radians, sin, cos, sqrt, asin
from collections import defaultdict
import re
from difflib import SequenceMatcher

NAME_MIN_TRIPS = 50
PAIR_RATIO_THRESHOLD = 0.80
PAIR_PARTIAL_THRESHOLD = 0.85
PAIR_TOKEN_OVERLAP_THRESHOLD = 0.80
TOKEN_MIN_LEN = 3
COMMON_TOKENS = {
    'de', 'du', 'des', 'la', 'le', 'les', 'et', 'st', 'ste',
    'rue', 'avenue', 'av', 'blvd', 'boulevard', 'chemin', 'parc'
}


def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return 2 * r * asin(sqrt(a))


def full_ratio(a: str, b: str) -> float:
    return SequenceMatcher(None, a, b).ratio()


def partial_ratio_fast(a: str, b: str) -> float:
    a = a.strip().lower()
    b = b.strip().lower()

    if not a or not b:
        return 0.0

    shorter, longer = (a, b) if len(a) <= len(b) else (b, a)

    if shorter in longer:
        return 1.0

    matcher = SequenceMatcher(None, shorter, longer)
    best = 0.0

    for block in matcher.get_matching_blocks():
        long_start = max(0, block[1] - block[0])
        window = longer[long_start:long_start + len(shorter)]
        if not window:
            continue
        score = SequenceMatcher(None, shorter, window).ratio()
        if score > best:
            best = score
        if best >= 0.995:
            break

    return best


def token_overlap_coeff(a: str, b: str) -> float:
    token_pattern = r"[a-z0-9]+"
    tokens_a = set(re.findall(token_pattern, a.lower()))
    tokens_b = set(re.findall(token_pattern, b.lower()))

    if not tokens_a or not tokens_b:
        return 0.0

    overlap = len(tokens_a & tokens_b)
    return overlap / min(len(tokens_a), len(tokens_b))


def blocking_tokens(name: str):
    tokens = re.findall(r"[a-z0-9]+", name.lower())
    tokens = [t for t in tokens if len(t) >= TOKEN_MIN_LEN and t not in COMMON_TOKENS]
    return sorted(set(tokens))


name_profile_query = """
WITH stations AS (
    SELECT
        LOWER(
            TRIM(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(
                        REGEXP_REPLACE(start_station_name, '\\s+', ' ', 'g'),
                        '\\s*/\\s*', ' / ', 'g'
                    ),
                    '\\s*-\\s*', '-', 'g'
                )
            )
        ) AS normalized_name,
        ROUND(CAST(start_station_latitude AS DOUBLE), 6) AS lat,
        ROUND(CAST(start_station_longitude AS DOUBLE), 6) AS lon,
        COUNT(*) AS coord_trips
    FROM read_parquet(?)
    WHERE start_station_name IS NOT NULL
      AND start_station_latitude IS NOT NULL
      AND start_station_longitude IS NOT NULL
    GROUP BY 1, 2, 3
), ranked AS (
    SELECT
        normalized_name,
        lat,
        lon,
        coord_trips,
        SUM(coord_trips) OVER (PARTITION BY normalized_name) AS total_name_trips,
        COUNT(*) OVER (PARTITION BY normalized_name) AS coord_count,
        ROW_NUMBER() OVER (PARTITION BY normalized_name ORDER BY coord_trips DESC, lat, lon) AS rn
    FROM stations
)
SELECT
    normalized_name,
    lat AS primary_lat,
    lon AS primary_lon,
    coord_trips AS primary_coord_trips,
    total_name_trips,
    coord_count
FROM ranked
WHERE rn = 1
  AND total_name_trips >= ?
ORDER BY total_name_trips DESC
"""

name_profile_df = duckdb.sql(name_profile_query, params=[PARQUET_GLOB, NAME_MIN_TRIPS]).to_df()

# Build candidate pairs via token blocking (avoids O(n^2) full pair scan)
name_records = list(name_profile_df.itertuples(index=False))
token_to_indices = defaultdict(list)
for idx, rec in enumerate(name_records):
    for tok in blocking_tokens(rec.normalized_name):
        token_to_indices[tok].append(idx)

candidate_pairs = set()
for idxs in token_to_indices.values():
    if len(idxs) < 2:
        continue
    for left_idx, right_idx in combinations(idxs, 2):
        if left_idx < right_idx:
            candidate_pairs.add((left_idx, right_idx))
        else:
            candidate_pairs.add((right_idx, left_idx))

pair_rows = []
for left_idx, right_idx in candidate_pairs:
    left_row = name_records[left_idx]
    right_row = name_records[right_idx]

    if left_row.primary_lat == right_row.primary_lat and left_row.primary_lon == right_row.primary_lon:
        continue

    left_name = left_row.normalized_name
    right_name = right_row.normalized_name

    ratio_score = full_ratio(left_name, right_name)
    partial_score = partial_ratio_fast(left_name, right_name)
    overlap_score = token_overlap_coeff(left_name, right_name)

    if (
        ratio_score >= PAIR_RATIO_THRESHOLD
        or partial_score >= PAIR_PARTIAL_THRESHOLD
        or overlap_score >= PAIR_TOKEN_OVERLAP_THRESHOLD
    ):
        distance_km = haversine_km(
            left_row.primary_lat,
            left_row.primary_lon,
            right_row.primary_lat,
            right_row.primary_lon,
        )
        pair_rows.append(
            {
                'left_name': left_name,
                'right_name': right_name,
                'ratio_score': round(ratio_score, 4),
                'partial_ratio_score': round(partial_score, 4),
                'token_overlap_score': round(overlap_score, 4),
                'max_score': round(max(ratio_score, partial_score, overlap_score), 4),
                'left_primary_lat': left_row.primary_lat,
                'left_primary_lon': left_row.primary_lon,
                'right_primary_lat': right_row.primary_lat,
                'right_primary_lon': right_row.primary_lon,
                'distance_km': round(distance_km, 4),
                'left_total_trips': int(left_row.total_name_trips),
                'right_total_trips': int(right_row.total_name_trips),
                'left_coord_count': int(left_row.coord_count),
                'right_coord_count': int(right_row.coord_count),
            }
        )

similar_name_diff_location_df = pd.DataFrame(pair_rows)

print(f"Station names profiled (min {NAME_MIN_TRIPS} trips): {len(name_profile_df):,}")
print(f"Candidate pairs generated after blocking: {len(candidate_pairs):,}")

if similar_name_diff_location_df.empty:
    print("No similar-name pairs found across different coordinates with current thresholds.")
else:
    print(f"Similar-name pairs at different coordinates: {len(similar_name_diff_location_df):,}")

    near_pairs = (similar_name_diff_location_df['distance_km'] <= 0.20).sum()
    mid_pairs = ((similar_name_diff_location_df['distance_km'] > 0.20) & (similar_name_diff_location_df['distance_km'] <= 1.0)).sum()
    far_pairs = (similar_name_diff_location_df['distance_km'] > 1.0).sum()

    print(f"Distance buckets -> <=0.2km: {near_pairs:,}, 0.2-1.0km: {mid_pairs:,}, >1.0km: {far_pairs:,}")

    display(
        similar_name_diff_location_df
        .sort_values(['max_score', 'distance_km'], ascending=[False, True])
        .head(80)
    )

Station names profiled (min 50 trips): 1,459
Candidate pairs generated after blocking: 23,498
Similar-name pairs at different coordinates: 888
Distance buckets -> <=0.2km: 229, 0.2-1.0km: 248, >1.0km: 411


,left_name,right_name,ratio_score,partial_ratio_score,token_overlap_score,max_score,left_primary_lat,left_primary_lon,right_primary_lat,right_primary_lon,distance_km,left_total_trips,right_total_trips,left_coord_count,right_coord_count
353,bélanger / des galeries d'anjou,des galeries d'anjou / bélanger,0.6452,0.6452,1.0,1.0,45.597221,-73.562790,45.597218,-73.562790,0.0003,4006,3887,1,2
28,milton / hutchison,milton / hutchison (ouest),0.8182,1.0000,1.0,1.0,45.508720,-73.574333,45.508728,-73.574326,0.0010,22544,2948,3,2
822,mcgill / place d'youville,place d'youville / mcgill,0.6400,0.6400,1.0,1.0,45.500015,-73.556252,45.500023,-73.556259,0.0010,42411,4026,1,3
168,lucien-l'allier / st-jacques,lucien l'allier / st-jacques,0.9643,0.9643,1.0,1.0,45.493526,-73.568520,45.493546,-73.568535,0.0025,12863,5266,1,1
314,métro assomption (chauveau / de l'assomption),chauveau / assomption,0.3333,0.7619,1.0,1.0,45.569790,-73.548080,45.569767,-73.548058,0.0031,7802,246,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
657,rachel / émile-duployé,s émile-duployé / rachel,0.6087,0.6364,1.0,1.0,45.529984,-73.569778,45.529709,-73.569984,0.0345,2762,753,1,1
335,3e avenue / wellington,wellington / 3e avenue,0.4545,0.4545,1.0,1.0,45.457306,-73.567230,45.457493,-73.567589,0.0349,24293,15125,3,2
47,de maisonneuve / de bleury,de bleury / de maisonneuve,0.5385,0.7000,1.0,1.0,45.507099,-73.569077,45.507107,-73.568611,0.0363,51953,23813,2,1
440,hickson / wellington,wellington / hickson,0.5000,0.5000,1.0,1.0,45.464825,-73.567123,45.465046,-73.566757,0.0377,33383,322,3,1


In [19]:
# Conclusion rule: stations within 0.05 km are approximately the same station -> group together
from collections import defaultdict

SAME_STATION_KM = 0.05

if 'similar_name_diff_location_df' not in globals() or similar_name_diff_location_df.empty:
    print("No similar-name different-location pairs found. Run the previous analysis cell first.")
else:
    same_station_edges_df = (
        similar_name_diff_location_df
        .loc[similar_name_diff_location_df['distance_km'] <= SAME_STATION_KM, ['left_name', 'right_name', 'distance_km']]
        .drop_duplicates()
        .copy()
    )

    if same_station_edges_df.empty:
        print(f"No pairs within {SAME_STATION_KM:.2f} km to group.")
    else:
        adjacency = defaultdict(set)
        for row in same_station_edges_df.itertuples(index=False):
            adjacency[row.left_name].add(row.right_name)
            adjacency[row.right_name].add(row.left_name)

        visited = set()
        groups = []

        for node in adjacency:
            if node in visited:
                continue
            stack = [node]
            component = []
            visited.add(node)

            while stack:
                current = stack.pop()
                component.append(current)
                for nei in adjacency[current]:
                    if nei not in visited:
                        visited.add(nei)
                        stack.append(nei)

            groups.append(sorted(component))

        groups = sorted(groups, key=lambda g: (-len(g), g[0]))

        station_group_rows = []
        for group_idx, names in enumerate(groups, start=1):
            canonical_name = names[0]
            group_id = f"station_group_{group_idx:04d}"
            for n in names:
                station_group_rows.append(
                    {
                        'group_id': group_id,
                        'canonical_name': canonical_name,
                        'member_name': n,
                        'group_size': len(names),
                    }
                )

        station_group_mapping_df = pd.DataFrame(station_group_rows)
        station_groups_summary_df = (
            station_group_mapping_df
            .groupby(['group_id', 'canonical_name', 'group_size'], as_index=False)
            .agg(member_names=('member_name', lambda s: sorted(s.tolist())))
            .sort_values(['group_size', 'group_id'], ascending=[False, True])
        )

        print(f"Pairs within {SAME_STATION_KM:.2f} km: {len(same_station_edges_df):,}")
        print(f"Grouped names (unique): {station_group_mapping_df['member_name'].nunique():,}")
        print(f"Number of station groups: {len(station_groups_summary_df):,}")

        display(station_groups_summary_df.head(50))

Pairs within 0.05 km: 115
Grouped names (unique): 192
Number of station groups: 87


,group_id,canonical_name,group_size,member_names
0,station_group_0001,parc émilie-gamelin (st-hubert / de maisonneuv...,5,[parc émilie-gamelin (st-hubert / de maisonneu...
1,station_group_0002,de kent / hudson,4,"[de kent / hudson, hudson / de kent, kent / hu..."
2,station_group_0003,de maisonneuve / mansfield (nord),4,"[de maisonneuve / mansfield (nord), de maisonn..."
3,station_group_0004,8e avenue / beaubien,3,"[8e avenue / beaubien, beaubien / 8e avenue, p..."
4,station_group_0005,calixa-lavallée / rachel,3,"[calixa-lavallée / rachel, calixa-lavallée / r..."
5,station_group_0006,chauveau / assomption,3,"[chauveau / assomption, métro assomption (chau..."
6,station_group_0007,de bleury / de maisonneuve,3,"[de bleury / de maisonneuve, de maisonneuve / ..."
7,station_group_0008,de l'hôtel-de-ville / du mont-royal,3,"[de l'hôtel-de-ville / du mont-royal, hotel-de..."
8,station_group_0009,fabre / jean-talon,3,"[fabre / jean-talon, métro fabre (fabre / jean..."
9,station_group_0010,louis-hémon / jean-talon,3,"[louis-hémon / jean-talon, parc gabriel-sagard..."


## Task 3: Build Canonical Station Mapping Table

### Objective
Create a complete `station_canonical_mapping_df` with canonical station IDs that consolidates:
- exact same-coordinate name variants,
- fuzzy similar names within 0.05 km,
- unchanged stations.

### Plan
1. Build station observations by `(raw_name, normalized_name, lat, lon)` with trip counts.
2. Assign each coordinate to a fuzzy group when applicable (from previous 0.05km grouping).
3. Select one canonical coordinate anchor per fuzzy group (highest traffic).
4. Generate canonical station IDs and map all station observations to them.

In [20]:
# Task 3 implementation: complete station_canonical_mapping table

# 1) Station observations
station_obs_query = """
SELECT
    start_station_name AS raw_name,
    LOWER(
        TRIM(
            REGEXP_REPLACE(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(start_station_name, '\\s+', ' ', 'g'),
                    '\\s*/\\s*', ' / ', 'g'
                ),
                '\\s*-\\s*', '-', 'g'
            )
        )
    ) AS normalized_name,
    ROUND(CAST(start_station_latitude AS DOUBLE), 6) AS lat,
    ROUND(CAST(start_station_longitude AS DOUBLE), 6) AS lon,
    COUNT(*) AS trip_count,
    MIN(EXTRACT(YEAR FROM to_timestamp(start_time_ms / 1000.0))) AS first_year_seen,
    MAX(EXTRACT(YEAR FROM to_timestamp(start_time_ms / 1000.0))) AS last_year_seen
FROM read_parquet(?)
WHERE start_station_name IS NOT NULL
  AND start_station_latitude IS NOT NULL
  AND start_station_longitude IS NOT NULL
GROUP BY 1, 2, 3, 4
"""

station_obs_df = duckdb.sql(station_obs_query, params=[PARQUET_GLOB]).to_df()
station_obs_df['coord_key'] = station_obs_df['lat'].map(lambda v: f"{v:.6f}") + "," + station_obs_df['lon'].map(lambda v: f"{v:.6f}")

# 2) Coordinate-level totals
coord_stats_df = (
    station_obs_df
    .groupby(['coord_key', 'lat', 'lon'], as_index=False)
    .agg(coord_total_trips=('trip_count', 'sum'))
)

# 3) Map normalized names to fuzzy group IDs (from Cell 7 result)
if 'station_group_mapping_df' in globals() and not station_group_mapping_df.empty:
    name_group_df = station_group_mapping_df[['group_id', 'member_name']].drop_duplicates().rename(columns={'member_name': 'normalized_name'})
else:
    name_group_df = pd.DataFrame(columns=['group_id', 'normalized_name'])

obs_with_group_df = station_obs_df.merge(name_group_df, how='left', on='normalized_name')

# 4) Assign dominant fuzzy group per coordinate (if any)
coord_group_votes_df = (
    obs_with_group_df[obs_with_group_df['group_id'].notna()]
    .groupby(['coord_key', 'group_id'], as_index=False)
    .agg(group_trip_votes=('trip_count', 'sum'))
)

if coord_group_votes_df.empty:
    coord_group_best_df = pd.DataFrame(columns=['coord_key', 'group_id'])
else:
    coord_group_best_df = (
        coord_group_votes_df
        .sort_values(['coord_key', 'group_trip_votes', 'group_id'], ascending=[True, False, True])
        .drop_duplicates(['coord_key'])
        [['coord_key', 'group_id']]
    )

coord_with_group_df = coord_stats_df.merge(coord_group_best_df, how='left', on='coord_key')

# 5) Choose canonical coordinate anchor per fuzzy group
if coord_with_group_df['group_id'].notna().any():
    group_anchor_df = (
        coord_with_group_df[coord_with_group_df['group_id'].notna()]
        .sort_values(['group_id', 'coord_total_trips', 'coord_key'], ascending=[True, False, True])
        .drop_duplicates(['group_id'])
        [['group_id', 'coord_key']]
        .rename(columns={'coord_key': 'canonical_coord_key'})
    )
else:
    group_anchor_df = pd.DataFrame(columns=['group_id', 'canonical_coord_key'])

coord_with_anchor_df = coord_with_group_df.merge(group_anchor_df, how='left', on='group_id')
coord_with_anchor_df['canonical_coord_key'] = coord_with_anchor_df['canonical_coord_key'].fillna(coord_with_anchor_df['coord_key'])

# 6) Build canonical station dimension and IDs
canonical_dim_df = (
    coord_with_anchor_df
    .groupby('canonical_coord_key', as_index=False)
    .agg(canonical_total_trips=('coord_total_trips', 'sum'))
    .sort_values(['canonical_total_trips', 'canonical_coord_key'], ascending=[False, True])
    .reset_index(drop=True)
)
canonical_dim_df['canonical_station_id'] = canonical_dim_df.index.map(lambda i: f"STN_{i+1:04d}")

# parse canonical coordinates
canonical_dim_df[['canonical_lat', 'canonical_lon']] = canonical_dim_df['canonical_coord_key'].str.split(',', expand=True)
canonical_dim_df['canonical_lat'] = canonical_dim_df['canonical_lat'].astype(float)
canonical_dim_df['canonical_lon'] = canonical_dim_df['canonical_lon'].astype(float)

# 7) Final mapping: each observed station variant -> canonical station ID
coord_to_canonical_df = coord_with_anchor_df[['coord_key', 'group_id', 'canonical_coord_key']].drop_duplicates()

station_canonical_mapping_df = (
    station_obs_df
    .merge(coord_to_canonical_df, how='left', on='coord_key')
    .merge(canonical_dim_df[['canonical_coord_key', 'canonical_station_id', 'canonical_lat', 'canonical_lon', 'canonical_total_trips']],
           how='left', on='canonical_coord_key')
)

station_canonical_mapping_df = station_canonical_mapping_df[[
    'canonical_station_id',
    'canonical_coord_key',
    'canonical_lat',
    'canonical_lon',
    'group_id',
    'raw_name',
    'normalized_name',
    'lat',
    'lon',
    'coord_key',
    'trip_count',
    'first_year_seen',
    'last_year_seen',
    'canonical_total_trips'
]].sort_values(['canonical_station_id', 'trip_count'], ascending=[True, False])

# 8) Station-level canonical summary
station_canonical_summary_df = (
    station_canonical_mapping_df
    .groupby(['canonical_station_id', 'canonical_coord_key', 'canonical_lat', 'canonical_lon'], as_index=False)
    .agg(
        canonical_total_trips=('trip_count', 'sum'),
        raw_name_variants=('raw_name', 'nunique'),
        normalized_name_variants=('normalized_name', 'nunique'),
        coord_variants=('coord_key', 'nunique'),
        first_year_seen=('first_year_seen', 'min'),
        last_year_seen=('last_year_seen', 'max')
    )
    .sort_values('canonical_total_trips', ascending=False)
)

print(f"Raw observed station variants (name+coord): {len(station_obs_df):,}")
print(f"Unique coordinates before canonicalization: {station_obs_df['coord_key'].nunique():,}")
print(f"Canonical stations after mapping: {station_canonical_summary_df['canonical_station_id'].nunique():,}")
print(f"Mappings assigned: {len(station_canonical_mapping_df):,}")

display(station_canonical_summary_df.head(30))
display(station_canonical_mapping_df.head(30))

Raw observed station variants (name+coord): 1,922
Unique coordinates before canonicalization: 1,730
Canonical stations after mapping: 1,577
Mappings assigned: 1,922


,canonical_station_id,canonical_coord_key,canonical_lat,canonical_lon,canonical_total_trips,raw_name_variants,normalized_name_variants,coord_variants,first_year_seen,last_year_seen
0,STN_0001,"45.519409,-73.586853",45.519409,-73.586853,231039,1,1,1,2024,2026
1,STN_0002,"45.532219,-73.575432",45.532219,-73.575432,174474,2,2,2,2024,2026
2,STN_0003,"45.527153,-73.589439",45.527153,-73.589439,163233,1,1,1,2024,2026
3,STN_0004,"45.515228,-73.575096",45.515228,-73.575096,153618,1,1,1,2024,2026
4,STN_0005,"45.502293,-73.573303",45.502293,-73.573303,136464,4,4,3,2024,2026
5,STN_0006,"45.500381,-73.575073",45.500381,-73.575073,128124,1,1,1,2024,2026
6,STN_0007,"45.524353,-73.581429",45.524353,-73.581429,124431,1,1,1,2025,2025
7,STN_0008,"45.503704,-73.569412",45.503704,-73.569412,122457,2,2,3,2024,2026
8,STN_0009,"45.527195,-73.564522",45.527195,-73.564522,119224,1,1,1,2024,2026
9,STN_0010,"45.489525,-73.584457",45.489525,-73.584457,117920,2,2,3,2024,2026


,canonical_station_id,canonical_coord_key,canonical_lat,canonical_lon,group_id,raw_name,normalized_name,lat,lon,coord_key,trip_count,first_year_seen,last_year_seen,canonical_total_trips
405,STN_0001,"45.519409,-73.586853",45.519409,-73.586853,NaN,du Mont-Royal / Clark,du mont-royal / clark,45.519409,-73.586853,"45.519409,-73.586853",231039,2024,2026,231039
1382,STN_0002,"45.532219,-73.575432",45.532219,-73.575432,station_group_0057,Marquette / du Mont-Royal,marquette / du mont-royal,45.532219,-73.575432,"45.532219,-73.575432",126015,2024,2025,174474
330,STN_0002,"45.532219,-73.575432",45.532219,-73.575432,station_group_0057,Marquette / du Mont-Royal (sud),marquette / du mont-royal (sud),45.532078,-73.575142,"45.532078,-73.575142",48459,2025,2026,174474
1599,STN_0003,"45.527153,-73.589439",45.527153,-73.589439,NaN,Laurier / St-Denis,laurier / st-denis,45.527153,-73.589439,"45.527153,-73.589439",163233,2024,2026,163233
750,STN_0004,"45.515228,-73.575096",45.515228,-73.575096,NaN,des Pins / St-Laurent,des pins / st-laurent,45.515228,-73.575096,"45.515228,-73.575096",153618,2024,2026,153618
1592,STN_0005,"45.502293,-73.573303",45.502293,-73.573303,station_group_0003,de Maisonneuve / Mansfield (sud-est),de maisonneuve / mansfield (sud-est),45.502293,-73.573303,"45.502293,-73.573303",55641,2024,2025,136464
1252,STN_0005,"45.502293,-73.573303",45.502293,-73.573303,station_group_0003,de Maisonneuve / Mansfield (sud),de maisonneuve / mansfield (sud),45.502052,-73.573463,"45.502052,-73.573463",50489,2024,2026,136464
1250,STN_0005,"45.502293,-73.573303",45.502293,-73.573303,station_group_0003,de Maisonneuve / Mansfield (nord),de maisonneuve / mansfield (nord),45.502384,-73.573341,"45.502384,-73.573341",29005,2024,2026,136464
1228,STN_0005,"45.502293,-73.573303",45.502293,-73.573303,station_group_0003,REM McGill (de Maisonneuve / Mansfield),rem mcgill (de maisonneuve / mansfield),45.502293,-73.573303,"45.502293,-73.573303",1329,2025,2026,136464
1709,STN_0006,"45.500381,-73.575073",45.500381,-73.575073,NaN,Métro Peel (de Maisonneuve / Stanley),métro peel (de maisonneuve / stanley),45.500381,-73.575073,"45.500381,-73.575073",128124,2024,2026,128124


## Task 4: Canonical Mapping Validation

### Objective
Validate mapping quality with three checks:
1. Self-loop reduction under canonical station IDs.
2. Station-count consolidation impact.
3. Unresolved conflicts and mapping coverage.

### Outputs
- `task4_summary_df`
- `task4_conflict_names_df`
- `task4_route_selfloop_compare_df`

In [22]:
# Task 4 validation: self-loop reduction, consolidation impact, unresolved conflicts

# Build coordinate -> canonical lookup
coord_lookup_df = (
    station_canonical_mapping_df[['coord_key', 'canonical_station_id']]
    .drop_duplicates()
    .copy()
)

# Register mapping table for DuckDB joins
duckdb.register('coord_lookup_df', coord_lookup_df)

validation_query = """
WITH trips AS (
    SELECT
        LOWER(
            TRIM(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(
                        REGEXP_REPLACE(start_station_name, '\\s+', ' ', 'g'),
                        '\\s*/\\s*', ' / ', 'g'
                    ),
                    '\\s*-\\s*', '-', 'g'
                )
            )
        ) AS start_norm_name,
        LOWER(
            TRIM(
                REGEXP_REPLACE(
                    REGEXP_REPLACE(
                        REGEXP_REPLACE(end_station_name, '\\s+', ' ', 'g'),
                        '\\s*/\\s*', ' / ', 'g'
                    ),
                    '\\s*-\\s*', '-', 'g'
                )
            )
        ) AS end_norm_name,
        ROUND(CAST(start_station_latitude AS DOUBLE), 6) AS start_lat,
        ROUND(CAST(start_station_longitude AS DOUBLE), 6) AS start_lon,
        ROUND(CAST(end_station_latitude AS DOUBLE), 6) AS end_lat,
        ROUND(CAST(end_station_longitude AS DOUBLE), 6) AS end_lon
    FROM read_parquet(?)
    WHERE start_station_name IS NOT NULL
      AND end_station_name IS NOT NULL
      AND start_station_latitude IS NOT NULL
      AND start_station_longitude IS NOT NULL
      AND end_station_latitude IS NOT NULL
      AND end_station_longitude IS NOT NULL
), keyed AS (
    SELECT
        *,
        CONCAT(printf('%.6f', start_lat), ',', printf('%.6f', start_lon)) AS start_coord_key,
        CONCAT(printf('%.6f', end_lat), ',', printf('%.6f', end_lon)) AS end_coord_key
    FROM trips
), mapped AS (
    SELECT
        k.*,
        s.canonical_station_id AS start_canonical_id,
        e.canonical_station_id AS end_canonical_id
    FROM keyed k
    LEFT JOIN coord_lookup_df s ON k.start_coord_key = s.coord_key
    LEFT JOIN coord_lookup_df e ON k.end_coord_key = e.coord_key
)
SELECT
    COUNT(*) AS total_trips,
    SUM(CASE WHEN start_norm_name = end_norm_name THEN 1 ELSE 0 END) AS raw_name_self_loops,
    SUM(CASE WHEN start_canonical_id = end_canonical_id AND start_canonical_id IS NOT NULL THEN 1 ELSE 0 END) AS canonical_self_loops,
    SUM(CASE WHEN start_canonical_id IS NULL OR end_canonical_id IS NULL THEN 1 ELSE 0 END) AS unmapped_trips
FROM mapped
"""

route_validation_df = duckdb.sql(validation_query, params=[PARQUET_GLOB]).to_df()

# Consolidation metrics
raw_name_count = int(station_obs_df['raw_name'].nunique())
norm_name_count = int(station_obs_df['normalized_name'].nunique())
coord_count = int(station_obs_df['coord_key'].nunique())
canonical_count = int(station_canonical_summary_df['canonical_station_id'].nunique())

# Conflict checks
coord_to_ids_conflict = (
    station_canonical_mapping_df
    .groupby('coord_key', as_index=False)
    .agg(canonical_id_count=('canonical_station_id', 'nunique'))
)
coord_conflict_count = int((coord_to_ids_conflict['canonical_id_count'] > 1).sum())

name_to_ids_conflict = (
    station_canonical_mapping_df
    .groupby('normalized_name', as_index=False)
    .agg(canonical_id_count=('canonical_station_id', 'nunique'), total_trips=('trip_count', 'sum'))
)
task4_conflict_names_df = (
    name_to_ids_conflict[name_to_ids_conflict['canonical_id_count'] > 1]
    .sort_values(['canonical_id_count', 'total_trips'], ascending=[False, False])
)

# Self-loop comparison summary
total_trips = int(route_validation_df.loc[0, 'total_trips'])
raw_self_loops = int(route_validation_df.loc[0, 'raw_name_self_loops'])
canonical_self_loops = int(route_validation_df.loc[0, 'canonical_self_loops'])
unmapped_trips = int(route_validation_df.loc[0, 'unmapped_trips'])

task4_route_selfloop_compare_df = pd.DataFrame([
    {'metric': 'total_trips', 'value': total_trips},
    {'metric': 'raw_name_self_loops', 'value': raw_self_loops},
    {'metric': 'canonical_self_loops', 'value': canonical_self_loops},
    {'metric': 'self_loop_delta_raw_minus_canonical', 'value': raw_self_loops - canonical_self_loops},
    {'metric': 'unmapped_trips', 'value': unmapped_trips},
    {'metric': 'unmapped_rate_pct', 'value': round(100 * unmapped_trips / total_trips, 4) if total_trips else 0.0},
])

task4_summary_df = pd.DataFrame([
    {'metric': 'raw_unique_station_names', 'value': raw_name_count},
    {'metric': 'normalized_unique_station_names', 'value': norm_name_count},
    {'metric': 'unique_coordinates_before', 'value': coord_count},
    {'metric': 'canonical_station_count_after', 'value': canonical_count},
    {'metric': 'coordinate_reduction', 'value': coord_count - canonical_count},
    {'metric': 'coordinate_reduction_pct', 'value': round(100 * (coord_count - canonical_count) / coord_count, 2) if coord_count else 0.0},
    {'metric': 'coord_to_multiple_canonical_ids_conflicts', 'value': coord_conflict_count},
    {'metric': 'normalized_names_mapping_to_multiple_canonical_ids', 'value': int(len(task4_conflict_names_df))},
])

print("Task 4 validation complete.")
display(task4_summary_df)
display(task4_route_selfloop_compare_df)
display(task4_conflict_names_df.head(30))

Task 4 validation complete.


,metric,value
0,raw_unique_station_names,1572.00
1,normalized_unique_station_names,1524.00
2,unique_coordinates_before,1730.00
3,canonical_station_count_after,1577.00
4,coordinate_reduction,153.00
5,coordinate_reduction_pct,8.84
6,coord_to_multiple_canonical_ids_conflicts,0.00
7,normalized_names_mapping_to_multiple_canonical...,226.00


,metric,value
0,total_trips,2.746226e+07
1,raw_name_self_loops,1.068037e+06
2,canonical_self_loops,1.073149e+06
3,self_loop_delta_raw_minus_canonical,-5.112000e+03
4,unmapped_trips,1.480000e+02
5,unmapped_rate_pct,5.000000e-04


,normalized_name,canonical_id_count,total_trips
1379,st-jacques / mcgill,6,33126
120,bloomfield / bernard,5,42414
481,drummond / de maisonneuve,5,17558
1293,roy / st-andré,4,76514
1440,ste-émilie / sir-georges-etienne-cartier,4,43225
1276,rielle / wellington,4,26270
1256,regina / de verdun,4,25445
1330,st-andré / laurier,4,22778
30,3e avenue / bannantyne,4,7436
965,métro mont-royal (utilités publiques / rivard),3,249555


## Final Conclusion and Next-Step Data Engineering Plan

### What was completed in this notebook

#### 1) Station-name inconsistency assessment
- Identified coordinate locations with multiple station names and quantified affected trips.
- Confirmed station naming drift across years and measured fragmentation risk.

#### 2) Name normalization and fuzzy matching
- Applied normalization (case, spacing, punctuation consistency).
- Added combined similarity logic (`ratio`, `partial_ratio`, token overlap) to detect likely duplicates.
- Extended analysis to similar station names at different coordinates.

#### 3) Proximity-based grouping rule
- Applied business rule: stations within **0.05 km** are treated as approximately the same station.
- Created grouped station-name mapping via connected components.

#### 4) Canonical mapping build (Task 3)
- Produced `station_canonical_mapping_df` and `station_canonical_summary_df`.
- Consolidated raw station variants into canonical station IDs.

#### 5) Validation (Task 4)
- Measured consolidation impact (coordinate count reduction).
- Checked mapping consistency (no coordinate mapping to multiple canonical IDs).
- Measured coverage (very low unmapped trip rate).
- Surfaced unresolved conflicts where one normalized name still maps to multiple canonical stations.

### Key interpretation
- The mapping is strong and production-ready for a first version.
- Remaining conflicts are mostly true multi-location names or ambiguous naming patterns that need controlled business rules.
- Canonical self-loops are slightly higher than raw-name self-loops, which is expected when fragmented variants are merged.

---

### Next steps: Implement as a PySpark CSV cleaning pipeline (no code)

#### Phase A — Pipeline architecture and contracts
1. Define input contract for CSV schema (same columns, data types, null rules, timezone assumptions).
2. Define output contracts:
   - cleaned trip fact table,
   - canonical station dimension,
   - station name mapping table,
   - quality/audit report.
3. Define partition strategy (for example by year/month) and storage format for outputs.
4. Define run mode: full backfill vs incremental daily/monthly processing.

#### Phase B — Bronze ingestion (raw landing)
1. Ingest CSV files exactly as received, with minimal transformation.
2. Add ingestion metadata (source file, ingestion timestamp, batch ID).
3. Persist raw records in a bronze zone for traceability and replay.
4. Add schema drift detection and reject/alert rules for unexpected columns or severe type breakage.

#### Phase C — Silver standardization (deterministic cleaning)
1. Normalize station names using deterministic text rules from this notebook.
2. Standardize coordinate precision (rounding policy fixed at pipeline level).
3. Generate deterministic coordinate keys for start/end stations.
4. Compute standardized temporal fields (trip date, year, month) for downstream joins.
5. Track row-level data quality flags (missing station name, missing coordinates, invalid times).

#### Phase D — Canonical station resolution
1. Build/update canonical station mapping table from silver station observations.
2. Apply two-step resolution logic:
   - exact coordinate consolidation,
   - fuzzy-name + proximity grouping using the 0.05 km rule.
3. Assign stable canonical station IDs (deterministic ID strategy, not order-dependent).
4. Keep versioned mapping snapshots with effective dates to support reproducibility.
5. Maintain unresolved conflict table for manual review and future rule updates.

#### Phase E — Gold outputs
1. Produce cleaned trip fact table with canonical start/end station IDs.
2. Produce canonical station dimension with attributes:
   - canonical ID,
   - canonical coordinates,
   - member name variants,
   - first/last seen period,
   - quality status.
3. Produce route-level aggregates using canonical IDs.
4. Publish quality KPIs per run (coverage, conflict count, coordinate reduction, self-loop deltas).

#### Phase F — Orchestration, observability, and controls
1. Schedule pipeline with parameterized runs (date range, full/incremental).
2. Add data quality gates:
   - max unmapped rate threshold,
   - max conflict growth threshold,
   - required-row-count sanity checks.
3. Add lineage and audit outputs (run ID, mapping version, input/output row counts).
4. Add alerting on failed quality gates and schema drift.
5. Add rollback strategy by preserving previous mapping versions.

#### Phase G — Operating model
1. Establish a weekly or monthly conflict-triage process for unresolved names.
2. Promote approved conflict decisions into rule tables (override mappings).
3. Re-run incremental correction jobs when rules change.
4. Track trend metrics over time to ensure mapping stability.

### Suggested implementation order
1. Bronze ingestion + Silver deterministic normalization.
2. Canonical mapping table build with versioning.
3. Gold fact/dimension publication.
4. Quality gates and alerting.
5. Conflict governance workflow and rule overrides.